# 01 — Data Pipeline

## Task 1.2 — Open-Meteo weather fetch

Fetches hourly `temperature_2m` and `shortwave_radiation` (GHI) from the
Open-Meteo historical archive for a given location and date range.

In [ ]:
import ssl
import requests
import pandas as pd
from requests.adapters import HTTPAdapter

In [ ]:
class _WinCertAdapter(HTTPAdapter):
    """
    Mounts Windows system CA store so requests works behind corporate proxies.
    """
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.load_default_certs(ssl.Purpose.SERVER_AUTH)
        kwargs["ssl_context"] = ctx
        super().init_poolmanager(*args, **kwargs)

_session = requests.Session()
_session.mount("https://", _WinCertAdapter())


def fetch_weather(lat, lon, start_date, end_date, timezone) -> pd.DataFrame:
    """
    Returns hourly UTC-aware DataFrame indexed by timestamp,
       with columns: temperature_2m, shortwave_radiation (Global Horizontal Irradiance, W/m²).

    Data is fetched and stored in UTC regardless of the timezone arg.
    Convert to local time only for plotting — solar physics works in UTC + longitude.
    """
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,shortwave_radiation",
        "timezone": "UTC",  # always fetch UTC — avoids DST NonExistentTime/AmbiguousTime errors
    }
    resp = _session.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=30,
    )
    resp.raise_for_status()

    hourly = resp.json()["hourly"]
    df = pd.DataFrame({
        "temperature_2m":      hourly["temperature_2m"],
        # shortwave_radiation == GHI (Global Horizontal Irradiance, W/m²)
        # pvlib will decompose this into DNI + DHI components
        "shortwave_radiation": hourly["shortwave_radiation"],
    }, index=pd.to_datetime(hourly["time"], utc=True))  # tz-aware UTC in one step, no tz_localize
    df.index.name = "timestamp"
    return df

In [ ]:
# Austin, TX — single source of truth; Week 2 pvlib uses these same coords
AUSTIN_LAT = 30.2672
AUSTIN_LON  = -97.7431
LOCAL_TZ    = "America/Chicago"

In [ ]:
# Smoke test — full year to exercise DST transitions
df_weather = fetch_weather(
    lat=AUSTIN_LAT,
    lon=AUSTIN_LON,
    start_date="2018-01-01",
    end_date="2018-12-31",
    timezone=LOCAL_TZ,
)

assert df_weather.index.tz is not None,               "Index must be tz-aware"
assert not df_weather.index.has_duplicates,            "Duplicate timestamps (DST fold?)"
assert df_weather.index.is_monotonic_increasing,       "Index not sorted"

# Open-Meteo occasionally returns null for shortwave_radiation at range boundaries
n_nans = df_weather.isna().sum().sum()
if n_nans:
    print(f"WARNING: {n_nans} NaNs present — investigate")

print(f"Rows: {len(df_weather)} (expect 8760 for a non-leap year)")
print(f"Index dtype: {df_weather.index.dtype}")
print(df_weather.head())

## Task 1.3 — Merge skeleton (built and proven against a mock)

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# Mock weather: 48 h of hourly UTC data (2018-01-01 – 2018-01-02)
# ---------------------------------------------------------------------------
_mock_weather_idx = pd.date_range("2018-01-01", periods=48, freq="h", tz="UTC")
mock_weather_df = pd.DataFrame({
    "temperature_2m":      np.linspace(5, 15, 48),
    "shortwave_radiation": np.clip(np.sin(np.linspace(0, 4 * np.pi, 48)) * 400, 0, None),
}, index=_mock_weather_idx)
mock_weather_df.index.name = "timestamp"

# ---------------------------------------------------------------------------
# Mock energy: same window but with four deliberate defects:
#   (a) missing hour        — 2018-01-01 15:00 UTC dropped entirely
#   (b) NaN load value      — household_load_kwh at 2018-01-01 20:00 set to NaN
#   (c) out-of-range rows   — two rows outside the weather window (before and after)
#   (d) NaN PV value        — actual_pv_yield_kwh at 2018-01-01 10:00 set to NaN
#                             tests that the "PV stays NaN" policy applies to genuine
#                             measured-then-dropped readings, not just reindex-created gaps
# ---------------------------------------------------------------------------
_energy_idx = pd.date_range("2018-01-01", periods=48, freq="h", tz="UTC")
mock_energy_df = pd.DataFrame({
    "household_load_kwh":  np.random.default_rng(42).uniform(0.3, 1.2, 48),
    "actual_pv_yield_kwh": np.clip(np.sin(np.linspace(0, 4 * np.pi, 48)) * 2, 0, None),
}, index=_energy_idx)
mock_energy_df.index.name = "timestamp"

# (a) drop one hour
mock_energy_df = mock_energy_df.drop(pd.Timestamp("2018-01-01 15:00", tz="UTC"))

# (b) inject a NaN into load
mock_energy_df.loc[pd.Timestamp("2018-01-01 20:00", tz="UTC"), "household_load_kwh"] = np.nan

# (c) append two out-of-range rows
_extra = pd.DataFrame({
    "household_load_kwh":  [0.9, 0.8],
    "actual_pv_yield_kwh": [0.0, 0.0],
}, index=pd.to_datetime(["2017-12-31 23:00", "2018-01-03 00:00"], utc=True))
_extra.index.name = "timestamp"
mock_energy_df = pd.concat([mock_energy_df, _extra]).sort_index()

# (d) inject a NaN into PV at a row that exists (not a dropped/reindex gap)
mock_energy_df.loc[pd.Timestamp("2018-01-01 10:00", tz="UTC"), "actual_pv_yield_kwh"] = np.nan

print(f"mock_weather rows : {len(mock_weather_df)}")
print(f"mock_energy rows  : {len(mock_energy_df)}")  # print actual count, don't trust arithmetic
print(f"Hour 15:00 missing from energy: {pd.Timestamp('2018-01-01 15:00', tz='UTC') not in mock_energy_df.index}")

In [ ]:
def build_unified_frame(weather_df: pd.DataFrame, energy_df: pd.DataFrame) -> pd.DataFrame:
    """
    Align energy data onto weather data on a complete hourly UTC index.
    Returns one frame: timestamp index +
      [temperature_2m, shortwave_radiation, household_load_kwh, actual_pv_yield_kwh]

    Join direction: weather range is canonical.
      - Week 2 physics prediction requires a weather row for every output timestamp;
        we cannot produce a solar estimate without it, so the weather window drives the index.
      - Energy rows outside that window are irrelevant and dropped by reindex.
      - Week 3 training handles residual NaNs by dropping those rows at fit-time.

    Missing energy hours:
      - household_load_kwh  → interpolate(method="time", limit=2, limit_direction="both").
          limit=2: gaps of 1–2 hours are estimable from neighbors; longer gaps (sensor outage)
          are left as NaN and dropped at training time rather than fabricated.
          limit_direction="both": handles edge NaNs where one neighbor is missing (start/end
          of window misalignment with real Pecan Street data).
      - actual_pv_yield_kwh → left as NaN in all cases. Solar yield depends on actual cloud
          cover; interpolating it would fabricate training signal.
    """
    # 1. Canonical index: every hour in the weather window, no gaps
    canonical = pd.date_range(
        start=weather_df.index.min(),
        end=weather_df.index.max(),
        freq="h",
        tz="UTC",
    )

    # 2. Reindex both sources — out-of-range energy rows dropped, gaps become NaN
    weather_aligned = weather_df.reindex(canonical)
    energy_aligned  = energy_df.reindex(canonical)

    # 3. Impute load only; PV intentionally stays NaN
    energy_aligned["household_load_kwh"] = energy_aligned["household_load_kwh"].interpolate(
        method="time",
        limit=2,               # gaps > 2 h left as NaN — don't invent a half-day load curve
        limit_direction="both", # fill edge NaNs from the available neighbor
    )

    # 4. Combine and name index
    result = pd.concat([weather_aligned, energy_aligned], axis=1)
    result.index.name = "timestamp"
    return result

In [ ]:
result = build_unified_frame(mock_weather_df, mock_energy_df)

# Index integrity
assert result.index.tz is not None,               "Index must be tz-aware"
assert not result.index.has_duplicates,            "Duplicate timestamps"
assert result.index.is_monotonic_increasing,       "Index not sorted"
assert len(result) == len(mock_weather_df),        "Row count must match weather window"

# (c) Out-of-range energy rows must be gone
assert pd.Timestamp("2017-12-31 23:00", tz="UTC") not in result.index, "Pre-range row leaked in"
assert pd.Timestamp("2018-01-03 00:00", tz="UTC") not in result.index, "Post-range row leaked in"

# (a) Missing hour restored; load interpolated, PV stays NaN (reindex-created gap)
missing_ts = pd.Timestamp("2018-01-01 15:00", tz="UTC")
assert missing_ts in result.index,                                        "Missing hour not restored"
assert pd.notna(result.loc[missing_ts, "household_load_kwh"]),            "load not interpolated at missing hour"
assert pd.isna(result.loc[missing_ts, "actual_pv_yield_kwh"]),            "PV should stay NaN at missing hour"

# (b) Explicit NaN in load interpolated
nan_load_ts = pd.Timestamp("2018-01-01 20:00", tz="UTC")
assert pd.notna(result.loc[nan_load_ts, "household_load_kwh"]),           "load NaN not interpolated"

# (d) Explicit NaN in PV stays NaN — tests the policy on a measured-then-nulled reading,
#     not just a reindex-created gap
nan_pv_ts = pd.Timestamp("2018-01-01 10:00", tz="UTC")
assert pd.isna(result.loc[nan_pv_ts, "actual_pv_yield_kwh"]),             "PV NaN must survive (no interpolation)"

print(result.info())
print(f"\nNaNs per column:\n{result.isna().sum()}")
# expect: load=0, pv=2 (hour 15:00 reindex gap + hour 10:00 explicit NaN)

## Task 1.4 — Pecan Street loader: power → hourly energy

### Column audit (done on the actual Pecan Street schema before writing any code)

Pecan Street residential exports contain:

| column | meaning | units |
|--------|---------|-------|
| `use`   | **gross household consumption** — total power drawn by the home | kW |
| `solar` | PV generation | kW (positive = generating) |
| `grid`  | net grid import/export = `use − solar` | kW (negative = exporting) |

`grid` is **not** household load — it is net consumption after the solar offset.  
Using `grid` as load would under-report consumption during sunny hours and produce nonsense correlations with irradiance in Week 3.  
→ Use `use` for `household_load_kwh`, `solar` for `actual_pv_yield_kwh`.

### kW → kWh decision

`mean(kW over one hour) × 1 h = kWh`  
For evenly-spaced samples this equals the trapezoidal integral.  
`.sum()` on 60 one-minute kW readings returns `Σ kW`, dimensionally `kW·samples`, not `kWh` — it is **60× too large**.  
If the native sampling is irregular (Pecan Street 15-min data has gaps), the simple mean is still valid as long as the gaps are small relative to the hour; large gaps produce NaN via `min_count` guard (see function).

### Sign convention

`solar` in Pecan Street is stored as positive generation.  No sign flip needed.  
`grid` is ignored entirely.

In [ ]:
def _hourly_energy(power_kw: pd.Series, min_valid: int) -> pd.Series:
    """mean kW per hour = kWh; NaN if fewer than min_valid sub-hour samples present.

    Uses count-then-mask rather than resample().mean(min_count=...) because
    Resampler.mean() does not accept min_count — only .sum()/.prod() do.
    """
    g = power_kw.resample("1h")
    return g.mean().where(g.count() >= min_valid)


def _infer_freq_minutes(df: pd.DataFrame) -> int:
    """Return median sampling interval in minutes.

    Assumes 1-min or 15-min data (standard Pecan Street exports).
    Sub-minute data (some 15-second exports) returns 0 and falls back to 1,
    which sets the wrong min_valid threshold — add explicit handling if
    targeting those exports.
    """
    diffs = df.index.to_series().diff().dropna()
    median_minutes = int(diffs.median().total_seconds() // 60)
    return median_minutes if median_minutes > 0 else 1


def load_pecan_street(csv_path, home_id, tz_source) -> pd.DataFrame:
    """
    Load one Pecan Street home, resample power (kW) → hourly energy (kWh),
    return tz-aware UTC hourly DataFrame: [household_load_kwh, actual_pv_yield_kwh]
    ready to feed straight into build_unified_frame().

    Column mapping (Pecan Street schema):
      use   → household_load_kwh  (gross consumption; NOT grid, which is net of solar)
      solar → actual_pv_yield_kwh (positive = generating; no sign flip needed)

    kW → kWh: _hourly_energy() takes the mean kW per hour (= kWh) and masks any
      hour with fewer than min_valid sub-hour readings as NaN, so large sensor gaps
      do not silently produce a plausible-looking low mean.
    """
    # Peek at column names cheaply before loading the full (potentially large) file,
    # then read only what's needed.
    _header = pd.read_csv(csv_path, nrows=0).columns.tolist()
    _usecols = [c for c in ["dataid", "localminute", "use", "solar"] if c in _header]
    df = pd.read_csv(csv_path, usecols=_usecols, parse_dates=["localminute"])
    df = df[df["dataid"] == home_id].copy()
    df = df.set_index("localminute")

    # Timezone: many Dataport exports ship localminute in America/Chicago local time.
    # ambiguous="NaT" converts DST-fold timestamps to NaT instead of crashing —
    # real Pecan Street data has gaps/duplicate-minute artifacts that break "infer".
    # We then drop the NaT rows; they are handled as missing data by min_valid downstream.
    # nonexistent="shift_forward" absorbs the spring-forward gap without error.
    # Convert to UTC immediately, same discipline as fetch_weather.
    df.index = (
        df.index
        .tz_localize(tz_source, ambiguous="NaT", nonexistent="shift_forward")
        .tz_convert("UTC")
    )
    df.index.name = "timestamp"
    df = df[df.index.notna()].sort_index()   # drop NaT rows from DST fold

    # Homes without a solar circuit: structural zero, not a sensor dropout.
    # fillna(0) is deliberately NOT applied to existing NaN readings on homes
    # that do have panels — unknown generation is not zero generation.
    if "solar" not in df.columns:
        df["solar"] = 0.0

    freq = _infer_freq_minutes(df)
    min_valid = 45 if freq == 1 else 3   # 75% of 60 min-samples or 4 fifteen-min slots

    hourly = pd.DataFrame({
        "household_load_kwh":  _hourly_energy(df["use"],   min_valid),
        "actual_pv_yield_kwh": _hourly_energy(df["solar"], min_valid),
    })
    return hourly

In [ ]:
# ── Mock-resample proof ─────────────────────────────────────────────────────
# Three hours of 1-minute data with known values:
#   hour 0: 60 readings at 2.0 kW / 0.5 kW  → full hour  → 2.0 kWh / 0.5 kWh
#   hour 1: 60 readings at 3.0 kW / 1.0 kW  → full hour  → 3.0 kWh / 1.0 kWh
#   hour 2: 10 readings at 4.0 kW            → sparse (< 45 min_valid) → NaN

_full  = pd.date_range("2018-01-01 06:00", periods=120, freq="min", tz="UTC")  # hours 0 & 1
_spare = pd.date_range("2018-01-01 08:00", periods=10,  freq="min", tz="UTC")  # 10 of 60 in hour 2

mock_pecan = pd.DataFrame({
    "use":   ([2.0] * 60) + ([3.0] * 60),
    "solar": ([0.5] * 60) + ([1.0] * 60),
}, index=_full)

mock_sparse = pd.DataFrame({
    "use":   [4.0] * 10,
    "solar": [2.0] * 10,
}, index=_spare)

mock_all = pd.concat([mock_pecan, mock_sparse])

# Correct resample via _hourly_energy
correct = pd.DataFrame({
    "household_load_kwh":  _hourly_energy(mock_all["use"],   min_valid=45),
    "actual_pv_yield_kwh": _hourly_energy(mock_all["solar"], min_valid=45),
})

# Wrong: unbounded mean (no count guard) — sparse hour silently returns a value
wrong = pd.DataFrame({
    "load_no_guard": mock_all["use"].resample("1h").mean(),
})

# Wrong: sum — 60x too large
wrong_sum = mock_all["use"].resample("1h").sum()

print("=== Correct (_hourly_energy with min_valid=45) ===")
print(correct.to_string())
print("\n=== Wrong (mean, no count guard) ===")
print(wrong.to_string())
print("\n=== Wrong (sum, 60x too large) ===")
print(wrong_sum.to_string())

# Full-hour values are correct
assert correct["household_load_kwh"].iloc[0]  == 2.0,  "hour-0 load 2.0 kWh"
assert correct["household_load_kwh"].iloc[1]  == 3.0,  "hour-1 load 3.0 kWh"
assert correct["actual_pv_yield_kwh"].iloc[0] == 0.5,  "hour-0 solar 0.5 kWh"
assert correct["actual_pv_yield_kwh"].iloc[1] == 1.0,  "hour-1 solar 1.0 kWh"

# Sparse hour is masked to NaN, not silently returned
assert pd.isna(correct["household_load_kwh"].iloc[2]),  "sparse hour must be NaN"
assert pd.isna(correct["actual_pv_yield_kwh"].iloc[2]), "sparse solar must be NaN"

# Without the guard, mean would silently produce a value
assert pd.notna(wrong["load_no_guard"].iloc[2]),        "unguarded mean returns a value (the bug)"

# Sum is 60x too large for a full hour
assert wrong_sum.iloc[0] == 120.0,                      "sum 60x too large"

print("\nAll resample-math assertions passed.")